# Strategy In-Depth Analysis
This notebook loads the T+2 portfolio weights generated by `alpha_pipeline.py` and provides a granular analysis of strategy performance, including monthly attribution, stock-level PnL aggregated by half-years, and rolling risk metrics.

In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os
import datetime

# Import utils
import sys
sys.path.append(os.path.abspath('phase2_qrt_challenge/scripts'))
import utils

DATA_DIR = "stores"

In [7]:
# 1. Load Data
# We look at the post-2020 period up to today
start_date = "2020-01-01"

print("Loading data...")
t2_weights = pd.read_parquet(os.path.join(DATA_DIR, 't2_weights.parquet')).loc[start_date:]
returns = pd.read_parquet(os.path.join(DATA_DIR, 'returns.parquet')).loc[start_date:]
universe = pd.read_parquet(os.path.join(DATA_DIR, 'universe_5m.parquet')).loc[start_date:]

# Ensure dimensions match
returns = returns.reindex_like(t2_weights).fillna(0)
print(f"Data loaded. Date range: {t2_weights.index[0].date()} to {t2_weights.index[-1].date()}")

Loading data...
Data loaded. Date range: 2020-01-02 to 2026-05-13


In [8]:
# 2. Compute Granular PnLs
# QRT Guidelines: 2bps execution cost + 0.5% annualised financing cost on GMV

# Daily Gross PnL
gross_pnl_stock = t2_weights * returns
gross_pnl_daily = gross_pnl_stock.sum(axis=1)

# Daily Costs
traded = t2_weights.diff(1).abs().fillna(0)
book_value = t2_weights.abs()

execution_cost_stock = traded * 2e-4
financing_cost_stock = book_value * (0.005 / 252)

# Daily Net PnL (Stock Level & Overall)
net_pnl_stock = gross_pnl_stock - execution_cost_stock - financing_cost_stock
net_pnl_daily = net_pnl_stock.sum(axis=1)

print(f"Total Gross PnL: {gross_pnl_daily.sum():.4f}")
print(f"Total Net PnL: {net_pnl_daily.sum():.4f}")

Total Gross PnL: 0.1387
Total Net PnL: -0.2149


In [9]:
# 3. Monthly Performance Analysis
monthly_net_pnl = net_pnl_daily.resample('ME').sum()
monthly_gross_pnl = gross_pnl_daily.resample('ME').sum()

winning_months = monthly_net_pnl[monthly_net_pnl > 0]
losing_months = monthly_net_pnl[monthly_net_pnl <= 0]

print("=== Monthly Performance Summary ===")
print(f"Total Months: {len(monthly_net_pnl)}")
print(f"Winning Months: {len(winning_months)} ({len(winning_months)/len(monthly_net_pnl)*100:.1f}%)")
print(f"Losing Months: {len(losing_months)} ({len(losing_months)/len(monthly_net_pnl)*100:.1f}%)")
print(f"Average Monthly Net PnL: {monthly_net_pnl.mean():.4f}")
print(f"Best Month: {monthly_net_pnl.max():.4f} ({monthly_net_pnl.idxmax().date()})")
print(f"Worst Month: {monthly_net_pnl.min():.4f} ({monthly_net_pnl.idxmin().date()})")

# Plot Cumulative Returns
utils.plot_series_with_names(
    [gross_pnl_daily.cumsum(), net_pnl_daily.cumsum()],
    ["Gross PnL", "Net PnL"],
    title="Cumulative Strategy Returns",
    yaxis_title="Cumulative Return",
    xaxis_title="Date"
)

=== Monthly Performance Summary ===
Total Months: 77
Winning Months: 31 (40.3%)
Losing Months: 46 (59.7%)
Average Monthly Net PnL: -0.0028
Best Month: 0.0449 (2020-04-30)
Worst Month: -0.1798 (2020-11-30)


In [10]:
# 4. Stock-Level Attribution (Every 6 Months)
# Resample to 6 months intervals
semi_annual_pnl = net_pnl_stock.resample('6ME').sum()

print("=== Semi-Annual Stock Attributes ===")
for period in semi_annual_pnl.index:
    period_pnl = semi_annual_pnl.loc[period]
    # Filter out stocks with 0 PnL
    period_pnl = period_pnl[period_pnl != 0]
    
    if period_pnl.empty:
        continue
        
    top_10 = period_pnl.nlargest(10)
    bottom_10 = period_pnl.nsmallest(10)
    
    print(f"\nPeriod ending {period.date()} Top 10 Winners:")
    print(top_10)
    
    print(f"\nPeriod ending {period.date()} Bottom 10 Losers:")
    print(bottom_10)

=== Semi-Annual Stock Attributes ===

Period ending 2020-01-31 Top 10 Winners:
Ticker
NVAX    0.000724
INMD    0.000343
CHRS    0.000336
MGNI    0.000315
KRYS    0.000256
ACMR    0.000255
CYH     0.000251
GOSS    0.000244
JMIA    0.000238
BYND    0.000236
Name: 2020-01-31 00:00:00, dtype: float64

Period ending 2020-01-31 Bottom 10 Losers:
Ticker
FLNA   -0.000547
FCEL   -0.000493
DTIL   -0.000378
ZYME   -0.000370
AM     -0.000324
CHRD   -0.000310
EQT    -0.000289
RIG    -0.000282
CDLX   -0.000264
GLNG   -0.000255
Name: 2020-01-31 00:00:00, dtype: float64

Period ending 2020-07-31 Top 10 Winners:
Ticker
IVR     0.002360
CHRD    0.002257
PTEN    0.002190
TOON    0.002184
CHEF    0.002162
PUMP    0.001708
LYFT    0.001661
ACB     0.001540
TPC     0.001385
TRTX    0.001332
Name: 2020-07-31 00:00:00, dtype: float64

Period ending 2020-07-31 Bottom 10 Losers:
Ticker
NBR    -0.002172
CLB    -0.001643
BNTX   -0.001434
ARMK   -0.001341
PMT    -0.001266
LYG    -0.001203
OXLC   -0.001193
ABR    -

In [11]:
# 5. Rolling Risk Metrics (120-Day Window)
window = 120

# Rolling Net Sharpe
rolling_mean = net_pnl_daily.rolling(window=window).mean()
rolling_std = net_pnl_daily.rolling(window=window).std()
rolling_sharpe = (rolling_mean / rolling_std) * np.sqrt(252)

# Rolling Net Sortino
# Downside deviation limits standard deviation only to negative returns
downside_returns = net_pnl_daily.where(net_pnl_daily < 0, 0)
downside_variance = downside_returns.rolling(window=window).apply(lambda x: (x**2).sum() / window)
downside_std = downside_variance.apply(np.sqrt)
rolling_sortino = (rolling_mean / downside_std) * np.sqrt(252)

# Sample the metrics at semi-annual marks (June & December)
is_half_year_end = (net_pnl_daily.index.is_month_end) & (net_pnl_daily.index.month.isin([6, 12]))
semi_annual_metrics = pd.DataFrame({
    '120-Day Sharpe': rolling_sharpe[is_half_year_end],
    '120-Day Sortino': rolling_sortino[is_half_year_end]
})

print("=== Semi-Annual Rolling Metrics (120-Day Context) ===")
print(semi_annual_metrics.dropna())

# Plot Rolling Metrics
utils.plot_series_with_names(
    [rolling_sharpe, rolling_sortino],
    ["120-Day Vol Sharpe", "120-Day Downside Sortino"],
    title="Rolling Risk-Adjusted Returns",
    yaxis_title="Annualized Ratio",
    xaxis_title="Date"
)

=== Semi-Annual Rolling Metrics (120-Day Context) ===
            120-Day Sharpe  120-Day Sortino
Date                                       
2020-06-30        1.800193         3.735286
2020-12-31       -1.548797        -1.554698
2021-06-30       -1.716783        -2.116239
2021-12-31        0.423586         0.658946
2022-06-30        0.394377         0.726441
2023-06-30       -0.389741        -0.536509
2024-12-31       -2.130871        -2.942919
2025-06-30        2.010858         6.916294
2025-12-31       -0.580040        -0.870315
